In [ ]:
from pathlib import Path
import polars as pl
projectPath = Path.cwd().resolve()
while projectPath.parent != projectPath and not (projectPath / 'Data').exists():
    projectPath = projectPath.parent
dataPrepPath = projectPath / 'pipelines' / 'dataprep' / 'output'
primaryPath = projectPath / 'pipelines' / 'primary' / 'output'
outputPath = projectPath / 'pipelines' / 'robustness' / 'output'
outputPath.mkdir(parents=True, exist_ok=True)
evaluationRows = pl.read_parquet(dataPrepPath / 'evaluationRows.parquet')
predictions = pl.read_parquet(primaryPath / 'modelPredictions.parquet').filter(pl.col('sourceTable') == 'mlTable')
scoredRows = predictions.join(evaluationRows.select('rowId', 'ndvi', 'ndviRange'), on='rowId', how='left').with_columns((pl.col('prediction') - pl.col('ndvi')).alias('error'))
errorByNdviRange = scoredRows.group_by('ndviRange', 'model', 'featureSet', 'seed').agg(pl.col('ndvi').min().alias('minimumNdvi'), pl.col('ndvi').max().alias('maximumNdvi'), pl.col('error').abs().mean().alias('mae'), pl.col('error').pow(2).mean().sqrt().alias('rmse'), pl.col('error').mean().alias('bias'), pl.len().alias('rowCount')).sort('minimumNdvi', 'model', 'featureSet', 'seed')
errorByNdviRange.write_parquet(outputPath / 'errorByNdviRange.parquet')
assert errorByNdviRange.select(pl.col('ndviRange').n_unique()).item() == 6
assert errorByNdviRange.select(pl.col('rowCount').sum()).item() == predictions.height
errorByNdviRange
